# W13A · R 실습 A

현재 자료로 값을 예측한 뒤 코드를 실행하세요.

In [ ]:
# Setup. Advertising uses summary statistics from the companion data.
ad_fields <- strsplit("At Pt Et At.1 Pt.1", " ")[[1]]
ad_cor <- matrix(c(1, -0.3569538830678335, -0.1285210088426754, -0.139740194995107, -0.4959911856331961, -0.3569538830678335, 1, 0.06258872102214971, -0.3164654862970893, -0.2963644633379876, -0.1285210088426754, 0.06258872102214971, 1, -0.1664279411745599, 0.2081123651377036, -0.139740194995107, -0.3164654862970893, -0.1664279411745599, 1, -0.3577610467749312, -0.4959911856331961, -0.2963644633379876, 0.2081123651377036, -0.3577610467749312, 1), 5L, byrow = TRUE,
                 dimnames = list(ad_fields, ad_fields))
sales_cor <- setNames(c(-0.1704050760465737, 0.5401768621663774, 0.8108731825755273, -0.3051611443563373, -0.05203527359556251), ad_fields)
sales_to_pt_sd <- 8.600968036261435

aux_r2_summary <- function(target, corr = ad_cor) {
  rest <- setdiff(colnames(corr), target)
  rel <- corr[rest, target]
  drop(crossprod(rel, solve(corr[rest, rest, drop = FALSE], rel)))
}

fit_proxy <- function(noise, use_proxy = TRUE) {
  stopifnot(is.numeric(noise), length(noise) == 1L, noise > 0)
  d <- datasets::attitude
  set.seed(1123)
  fixed_z <- as.numeric(scale(rnorm(nrow(d))))
  d$proxy <- d$complaints + noise * sd(d$complaints) * fixed_z
  f <- if (use_proxy) rating ~ complaints + learning + proxy else rating ~ complaints + learning
  lm(f, data = d)
}

snapshot <- function(model) {
  scores <- coef(summary(model))
  bounds <- confint(model, "complaints")
  data.frame(R2 = summary(model)$r.squared,
             coefficient = scores["complaints", 1L],
             SE = scores["complaints", 2L],
             lower = bounds[1L], upper = bounds[2L])
}
# Companion-data route for RStudio and Colab.
read_advertising <- function() {
  data <- read.delim("https://www1.aucegypt.edu/faculty/hadi/RABE6/Data6/Advertising.txt", check.names = TRUE)
  stopifnot(nrow(data) == 22L, identical(names(data), c("St", ad_fields)))
  data
}

fit_ad_aux <- function(data, target) {
  rest <- setdiff(ad_fields, target)
  lm(reformulate(rest, response = target), data)
}

In [ ]:
# Q3/Q5. TODO: change At to Pt and compare the result.
ad <- read_advertising()
focus <- "At"
aux <- fit_ad_aux(ad, focus)
aux_r2 <- summary(aux)$r.squared
R <- cor(ad[ad_fields])
print(round(c(max_pair = max(abs(R[upper.tri(R)])),
              aux_R2 = aux_r2, VIF = 1 / (1 - aux_r2)), 4))

In [ ]:
stopifnot(identical(dim(ad_cor), c(5L, 5L)),
          isSymmetric(ad_cor), all(is.finite(snapshot(fit_proxy(1))$SE)),
          nobs(fit_proxy(.15)) == 30L)
cat("W13A_SETUP_OK\n")